# Cleaning Dataset
## Importing Libraries

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pyarrow

## Converting Month Datasets into Long Format
### Reading in All Datasets

In [2]:
# Load raw data
jan_wide = pd.read_parquet('../data/raw/Alta10_Turbine_Points_10m_2026-01.parquet')
feb_wide = pd.read_parquet('../data/raw/Alta10_Turbine_Points_10m_2026-02.parquet')
mar_wide = pd.read_parquet('../data/raw/Alta10_Turbine_Points_10m_2026-03.parquet')
apr_wide = pd.read_parquet('../data/raw/Alta10_Turbine_Points_10m_2026-04.parquet')
may_wide = pd.read_parquet('../data/raw/Alta10_Turbine_Points_10m_2026-05.parquet')
jun_wide = pd.read_parquet('../data/raw/Alta10_Turbine_Points_10m_2026-06.parquet')
jul_wide = pd.read_parquet('../data/raw/Alta10_Turbine_Points_10m_2026-07.parquet')
aug_wide = pd.read_parquet('../data/raw/Alta10_Turbine_Points_10m_2026-08.parquet')
sep_wide = pd.read_parquet('../data/raw/Alta10_Turbine_Points_10m_2026-09.parquet')

### Analyzing Missing Date Values

In [38]:
# defining a list of the monthly datasets
month = ['January', 'February', 'March', 'April', 'May', 'June', 'July', 'August', 'September']
monthly_wide = [jan_wide, feb_wide, mar_wide, apr_wide, may_wide, jun_wide, jul_wide, aug_wide, sep_wide]
exp_counts_wide = [4464, 4032, 4464, 4320, 4464, 4320, 4464, 4464, 4320]

# finding the number of observations below expected in each dataset
for n in range(9):
    print(month[n], ':', exp_counts_wide[n] - monthly_wide[n].shape[0])

January : 0
February : 0
March : 6
April : 0
May : 0
June : 0
July : 0
August : 0
September : 288


In [34]:
# defining a function to mark missing date/times in the datasets
def missing_times(df, year, month):
    # defines the starting time of 10 minute incriments
    start = pd.Timestamp(year = year, month = month, day = 1)
    end = start + pd.offsets.MonthBegin(1) - pd.Timedelta(minutes=10)
    expected = pd.date_range(start, end, freq = '10min')

    # calculates missing values
    missing = expected.difference(df.index)

    # returns missing datetimes
    return missing

In [32]:
# finding the missing times in March
missing_times(mar_wide, 2026, 3)

DatetimeIndex(['2026-03-08 02:00:00', '2026-03-08 02:10:00',
               '2026-03-08 02:20:00', '2026-03-08 02:30:00',
               '2026-03-08 02:40:00', '2026-03-08 02:50:00'],
              dtype='datetime64[us]', freq='10min')

The March dataset is missing an hour on March 8, 2026 from 2:00 AM to 2:50 AM. 

In [33]:
# finding the missing times in September
missing_times(sep_wide, 2026, 9)

DatetimeIndex(['2026-09-29 00:00:00', '2026-09-29 00:10:00',
               '2026-09-29 00:20:00', '2026-09-29 00:30:00',
               '2026-09-29 00:40:00', '2026-09-29 00:50:00',
               '2026-09-29 01:00:00', '2026-09-29 01:10:00',
               '2026-09-29 01:20:00', '2026-09-29 01:30:00',
               ...
               '2026-09-30 22:20:00', '2026-09-30 22:30:00',
               '2026-09-30 22:40:00', '2026-09-30 22:50:00',
               '2026-09-30 23:00:00', '2026-09-30 23:10:00',
               '2026-09-30 23:20:00', '2026-09-30 23:30:00',
               '2026-09-30 23:40:00', '2026-09-30 23:50:00'],
              dtype='datetime64[us]', length=288, freq='10min')

The September dataset is missing 2 entire days, September 29, 2026 and September 30, 2026. 
### Converting Datasets to Long Format

In [3]:
# reordering and renaming variables with a dictionary
mapping = {'utcTime': 'utc', 'turbine': 'turbine', 'AirDensity': 'aid_density', 'KPI-TemperatureAmbient-Merged': 'KPI_temp_ambient_merged', 
           'KPI-TemperatureAmbient': 'KPI_temp_ambient', 'KPI-TheoreticalPower': 'KPI_theoretical_power', 'KPI-RealPowerAC': 'KPI_real_power_ac',
           'RealPowerAC': 'real_power_ac', 'KPI-WindSpeed': 'KPI_wind_speed', 'WindSpeed': 'wind_speed', 'WindDirection': 'wind_direction', 
           'WindDirectionRelative': 'wind_direction_relative', 'NacellePosition': 'nacelle_pos', 'BladePosition1': 'blade_1_pos', 
           'BladePosition2': 'blade_2_pos', 'BladePosition3': 'blade_3_pos', 'BladeSetpointPosition1': 'blade_1_setpos', 
           'BladeSetpointPosition2': 'blade_2_setpos', 'BladeSetpointPosition3': 'blade_3_setpoint'}

# defining a function to convert the wide format datasets into long format datasets
def reshape_year(df):
    # defines utcTime as index for rows in new dataset
    df = df.set_index('utcTime')

    # splits the columns into the turbine name and variable name
    parts = df.columns.str.extract(r"^ALTA\d+-SCEWF-(WTG\d+)-(.+)$")
    parts.columns = ['turbine', 'variable']

    # outputs problematic columns from this split
    bad = df.columns[parts['turbine'].isna()]
    if len(bad):
        raise ValueError(f"Columns that didn't match the pattern: {list(bad)}")

    # creates a copy of the dataframe
    df = df.copy()
    df.columns = pd.MultiIndex.from_frame(parts)

    # creates the long version of the dataset
    long = df.stack(level = 'turbine', future_stack = True).reset_index()
    long['turbine'] = long['turbine'].astype(str)
    long.columns.name = None
    long.sort_values(['turbine', 'utcTime']).reset_index(drop=True)

    # returns new dataset with renamed and reordered columns
    return long[list(mapping)].rename(columns=mapping)

In [4]:
# applying the reshape_year() function to each monthly dataset
jan = reshape_year(jan_wide)
feb = reshape_year(feb_wide)
mar = reshape_year(mar_wide)
apr = reshape_year(apr_wide)
may = reshape_year(may_wide)
jun = reshape_year(jun_wide)
jul = reshape_year(jul_wide)
aug = reshape_year(aug_wide)
sep = reshape_year(sep_wide)

### Creating a Single Dataset with All Months

In [44]:
# creating a datasets stacking all monthly data
scada = pd.concat(monthly)

#### Looking at Missing Datetime Values in Transformed Data

In [39]:
# defining a list of the monthly datasets
month = ['January', 'February', 'March', 'April', 'May', 'June', 'July', 'August', 'September']
monthly = [jan, feb, mar, apr, may, jun, jul, aug, sep]
exp_counts = [209808, 189504, 209808, 203040, 209808, 203040, 209808, 209808, 203040]

# finding the number of observations below expected in each dataset
for n in range(9):
    print(month[n], ':', exp_counts[n] - monthly[n].shape[0])

January : 0
February : 0
March : 282
April : 0
May : 0
June : 0
July : 0
August : 0
September : 13536


The months of March and September have missing time values. However, this completely aligns with the missing time findings in the wide-format datasets.
### Missing Data Analysis

In [50]:
# analyzing which columns have the most missing data
na_pct = scada.isna().mean()

print(na_pct.sort_values(ascending = False))

blade_1_setpos             0.342165
blade_3_setpoint           0.342151
blade_2_setpos             0.342150
real_power_ac              0.341689
blade_1_pos                0.304447
blade_3_pos                0.289025
blade_2_pos                0.288620
nacelle_pos                0.220685
KPI_wind_speed             0.085207
KPI_temp_ambient           0.081829
KPI_real_power_ac          0.077143
wind_speed                 0.076460
wind_direction_relative    0.076432
wind_direction             0.076420
aid_density                0.005018
KPI_temp_ambient_merged    0.003934
KPI_theoretical_power      0.000371
turbine                    0.000000
utc                        0.000000
dtype: float64
